In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc matplotlib numpy
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     channel="ibm_quantum_platform",
#     token="<your-api-key>",
#     # instance="<IBM Cloud CRN or instance name>",  # optional
#     set_as_default=True,
#     overwrite=True,
# )

import TutorialFeedback from '@site/src/components/TutorialFeedback';





*Anggaran penggunaan: 4 minit pada pemproses Heron r2. (NOTA: Ini adalah anggaran sahaja. Masa sebenar mungkin berbeza.)*

## Hasil pembelajaran

Setelah menyelesaikan tutorial ini, anda akan mempelajari perkara berikut:
* Cara melaksanakan Gate CNOT jarak jauh menggunakan litar dinamik dengan pengukuran pertengahan litar (MCM) dan feedforward klasikal;
* Cara melaksanakan Gate setara menggunakan pendekatan berasaskan SWAP uniter;
* Cara membandingkan kedua-dua pendekatan dengan mengukur kesetiaan Gate sebagai fungsi jarak Qubit.

## Prasyarat

Kami cadangkan pengguna biasa dengan topik berikut sebelum melalui tutorial ini:
* [Konsep pengkomputeran kuantum asas](/learning/courses/basics-of-quantum-information), termasuk keadaan Bell, keterikatan, dan Gate kuantum;
* Kebiasaan dengan [litar dinamik](/guides/classical-feedforward-and-control-flow) (pengukuran pertengahan litar dan feedforward klasikal);
* Pengetahuan asas [Qiskit SDK](https://docs.quantum.ibm.com/guides) dan [Qiskit Runtime](/guides/compute-services#qiskit-runtime), serta akses kepada [akaun IBM Quantum&reg;](/guides/cloud-setup).

## Latar belakang

Keterikatan jarak jauh antara Qubit yang berjauhan adalah sukar pada peranti dengan sambungan terhad. Tutorial ini menunjukkan bagaimana litar dinamik boleh menjana keterikatan tersebut dengan melaksanakan Gate kawalan-X jarak jauh (LRCX) menggunakan protokol berasaskan pengukuran.

Mengikut pendekatan Elisa Bäumer et al. dalam [1](#ref-1), kaedah ini menggunakan pengukuran pertengahan litar dan feedforward untuk mencapai Gate kedalaman tetap tanpa mengira jarak pemisahan Qubit. Ia mencipta pasangan Bell perantara, mengukur satu Qubit dari setiap pasangan, dan menggunakan Gate bersyarat secara klasik untuk menyebarkan keterikatan merentasi peranti. Ini mengelakkan rantaian SWAP yang panjang, mengurangkan kedalaman litar dan pendedahan kepada ralat Gate dua Qubit.

Dalam buku nota ini, kami menyesuaikan protokol untuk perkakasan IBM Quantum dan membuat penanda aras prestasinya sebagai fungsi pemisahan kawalan–sasaran, membandingkannya dengan garis dasar berasaskan SWAP uniter.

## Keperluan

Sebelum memulakan tutorial ini, pastikan anda telah memasang perkara berikut:

- Qiskit SDK v2.0 atau lebih baharu, dengan sokongan [visualisasi](https://docs.quantum.ibm.com/api/qiskit/visualization)
- Qiskit Runtime v0.37 atau lebih baharu (`pip install qiskit-ibm-runtime`)
- Qiskit Aer v0.17 atau lebih baharu (`pip install qiskit-aer`)

## Persediaan

In [2]:
from qiskit import QuantumCircuit, QuantumRegister, ClassicalRegister
from qiskit.circuit.classical import expr
from qiskit.transpiler import generate_preset_pass_manager
from qiskit.visualization import plot_circuit_layout
from qiskit_ibm_runtime import (
    QiskitRuntimeService,
    Batch,
    SamplerV2 as Sampler,
)
import matplotlib.pyplot as plt
import numpy as np

## Contoh simulator skala kecil

Sebelum dijalankan pada QPU sebenar, kita sahkan bahawa kedua-dua litar dinamik dan uniter menghasilkan keadaan Bell yang ideal pada simulator tanpa hingar. Kita gunakan Qiskit Runtime `Sampler` dengan `AerSimulator` sebagai mod backend, pada jarak kecil iaitu 6.

### Langkah 1: Petakan input klasik kepada masalah kuantum

Kini kami melaksanakan Gate CNOT jarak jauh antara dua Qubit yang berjauhan, mengikut pembinaan litar dinamik yang ditunjukkan di bawah (disesuaikan daripada Rajah 1a dalam Ref. [1](#ref-1)). Idea utamanya adalah menggunakan "bas" Qubit ancilla, yang dimulakan kepada $|0\rangle$, untuk menjadi perantara teleportasi Gate jarak jauh.

![Litar CNOT jarak jauh](../docs/images/tutorials/long-range-entanglement/dynamic_vs_unitary_long_range_illustration.avif)

Seperti yang digambarkan dalam rajah, prosesnya berjalan seperti berikut:
1. Sediakan rantaian pasangan Bell yang menghubungkan Qubit kawalan dan sasaran melalui ancilla perantara.
2. Lakukan pengukuran Bell antara Qubit jiran yang tidak terikat, menukar keterikatan langkah demi langkah sehingga kawalan dan sasaran berkongsi pasangan Bell.
3. Gunakan pasangan Bell ini untuk teleportasi Gate, mengubah CNOT setempat menjadi CNOT jarak jauh yang deterministik pada kedalaman tetap.

Pendekatan ini menggantikan rantaian SWAP yang panjang dengan protokol kedalaman tetap, mengurangkan pendedahan kepada ralat Gate dua Qubit dan menjadikan operasi boleh berskala dengan saiz peranti.

Dalam bahagian berikut, kita akan melalui pelaksanaan litar dinamik bagi litar LRCX. Di akhir tutorial, kami juga akan menyediakan pelaksanaan berasaskan uniter untuk perbandingan, bagi menyerlahkan kelebihan litar dinamik dalam konteks ini.

#### Mulakan litar

Kita mulakan dengan masalah kuantum mudah yang akan menjadi asas perbandingan. Secara khusus, kita memulakan litar dengan Qubit kawalan pada indeks 0 dan menggunakan Gate Hadamard padanya. Ini menghasilkan keadaan superposisi yang, apabila diikuti dengan operasi kawalan-X, menghasilkan keadaan Bell $(|00\rangle + |11\rangle)/\sqrt{2}$ antara Qubit kawalan dan sasaran.

Pada tahap ini, kita belum lagi membina Gate kawalan-X jarak jauh (LRCX) itu sendiri. Sebaliknya, matlamat kita adalah untuk mentakrifkan litar permulaan yang jelas dan ringkas yang menonjolkan peranan LRCX. Dalam Langkah 2, kita akan menunjukkan bagaimana LRCX boleh dilaksanakan sebagai pengoptimuman menggunakan litar dinamik, dan membandingkan prestasinya dengan setara uniter. Penting untuk diingat, protokol LRCX boleh digunakan pada mana-mana litar permulaan. Di sini kita gunakan persediaan Hadamard mudah ini untuk kejelasan demonstrasi.

In [20]:
distance = 6  # The distance of the CNOT gate, with the convention that a distance of zero is a nearest-neighbor CNOT.


def initialize_circuit(distance):
    assert distance >= 0
    control = 0  # control qubit
    n = distance  # number of qubits between target and control

    qr = QuantumRegister(
        n + 2, name="q"
    )  # Circuit with n qubits between control and target
    cr = ClassicalRegister(
        2, name="cr"
    )  # Classical register for measuring control and target qubits

    k = int(n / 2)  # Number of Bell States to be used

    allcr = [cr]
    if (
        distance > 1
    ):  # This classical register will be used to store ZZ measurements. It is only used for long-range CX gates with distance > 1
        c1 = ClassicalRegister(
            k, name="c1"
        )  # Classical register needed for post processing
        allcr.append(c1)
    if (
        distance > 0
    ):  # This classical register will be used to store XX measurements. It is only used if distance > 0
        c2 = ClassicalRegister(
            n - k, name="c2"
        )  # Classical register needed for post processing
        allcr.append(c2)

    qc = QuantumCircuit(qr, *allcr, name="CNOT")

    # Apply a Hadamard gate to the control qubit such that the long-range CNOT gate will prepare a Bell state (|00> + |11>)/sqrt(2)
    qc.h(control)

    return qc


qc = initialize_circuit(distance)
qc.draw(fold=-1, output="mpl", scale=0.5)

<Image src="../docs/images/tutorials/long-range-entanglement/extracted-outputs/0446b8e8-0.avif" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/tutorials/long-range-entanglement/extracted-outputs/0446b8e8-0.avif)

### Langkah 2: Optimumkan masalah untuk pelaksanaan perkakasan kuantum
Dalam langkah ini, kita tunjukkan cara membina litar LRCX menggunakan litar dinamik. Matlamatnya adalah untuk mengoptimumkan litar bagi pelaksanaan pada perkakasan dengan mengurangkan kedalaman berbanding pelaksanaan uniter semata-mata. Untuk menggambarkan manfaatnya, kita akan memaparkan kedua-dua pembinaan LRCX dinamik dan setara uniternya, serta kemudian membandingkan prestasinya selepas transpilasi. Penting untuk diingat, walaupun di sini kita menggunakan LRCX pada masalah yang dimulakan dengan Hadamard yang mudah, protokol ini boleh digunakan pada mana-mana litar yang memerlukan CNOT jarak jauh.

#### Sediakan pasangan Bell
Kita mulakan dengan mencipta rantaian pasangan Bell di sepanjang laluan antara Qubit kawalan dan sasaran. Jika jaraknya ganjil, kita mula-mula menggunakan CNOT dari kawalan ke jirannya, iaitu CNOT yang akan diteleportkan. Bagi jarak genap, CNOT ini akan digunakan selepas langkah penyediaan pasangan Bell. Rantaian pasangan Bell kemudiannya mengikat pasangan Qubit yang berturutan, mewujudkan sumber yang diperlukan untuk membawa maklumat kawalan merentasi peranti.

In [4]:
# Determine where to start the Bell pair chain and add an extra CNOT when n is odd
def check_even(n: int) -> int:
    """Return 1 if n is even, else 2."""
    return 1 if n % 2 == 0 else 2


def prepare_bell_pairs(qc, add_barriers=True):
    n = qc.num_qubits - 2  # number of qubits between target and control
    k = int(n / 2)

    if add_barriers:
        qc.barrier()

    x0 = check_even(n)
    if n % 2 != 0:
        qc.cx(0, 1)

    # Create k Bell pairs
    for i in range(k):
        qc.h(x0 + 2 * i)
        qc.cx(x0 + 2 * i, x0 + 2 * i + 1)
    return qc


qc = prepare_bell_pairs(qc)
qc.draw(output="mpl", fold=-1, scale=0.5)

<Image src="../docs/images/tutorials/long-range-entanglement/extracted-outputs/4df8ebba-0.avif" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/tutorials/long-range-entanglement/extracted-outputs/4df8ebba-0.avif)

#### Ukur pasangan Qubit jiran dalam asas Bell
Seterusnya, kita ukur Qubit jiran yang *tidak terikat* dalam asas Bell (pengukuran dua Qubit bagi $XX$ dan $ZZ$). Ini mewujudkan pasangan Bell jarak jauh antara Qubit sasaran dan Qubit bersebelahan dengan kawalan (tertakluk kepada pembetulan Pauli, yang akan dilaksanakan melalui feedforward dalam langkah berikut). Secara selari, kita melaksanakan pengukuran pengikat yang menteleportkan Gate CNOT untuk bertindak pada Qubit sasaran yang dimaksudkan.

In [5]:
def measure_bell_basis(qc, add_barriers=True):
    n = qc.num_qubits - 2  # number of qubits between target and control
    k = int(n / 2)

    if n > 1:
        _, c1, c2 = qc.cregs
    elif n > 0:
        _, c2 = qc.cregs

    # Determine where to start the Bell pair chain and add an extra CNOT when n is odd
    x0 = 1 if n % 2 == 0 else 2

    # Entangling layer that implements the Bell measurement (and additionally adds the CNOT to be teleported, if n is even)
    for i in range(k + 1):
        qc.cx(x0 - 1 + 2 * i, x0 + 2 * i)

    for i in range(1, k + x0):
        if i == 1:
            qc.h(2 * i + 1 - x0)
        else:
            qc.h(2 * i + 1 - x0)

    if add_barriers:
        qc.barrier()

    # Map the ZZ measurements onto classical register c1
    for i in range(k):
        if i == 0:
            qc.measure(2 * i + x0, c1[i])
        else:
            qc.measure(2 * i + x0, c1[i])

    # Map the XX measurements onto classical register c2
    for i in range(1, k + x0):
        if i == 1:
            qc.measure(2 * i + 1 - x0, c2[i - 1])
        else:
            qc.measure(2 * i + 1 - x0, c2[i - 1])
    return qc


qc = measure_bell_basis(qc)
qc.draw(output="mpl", fold=-1, scale=0.5)

<Image src="../docs/images/tutorials/long-range-entanglement/extracted-outputs/8eed9e57-0.avif" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/tutorials/long-range-entanglement/extracted-outputs/8eed9e57-0.avif)

#### Gunakan pembetulan feedforward untuk membetulkan operator sampingan Pauli
Pengukuran asas Bell memperkenalkan produk sampingan Pauli yang perlu diperbetulkan menggunakan hasil ukuran yang direkodkan. Ini dilakukan dalam dua langkah. Pertama, kita perlu mengira pariti semua pengukuran $ZZ$, yang kemudian digunakan untuk menggunakan Gate $X$ secara bersyarat pada Qubit sasaran. Begitu juga, pariti pengukuran $XX$ dikira dan digunakan untuk menggunakan Gate $Z$ secara bersyarat pada Qubit kawalan.

Dengan rangka kerja ungkapan klasik baharu dalam Qiskit, pariti-pariti ini boleh dikira terus dalam lapisan pemprosesan klasik litar. Daripada menggunakan urutan Gate bersyarat individu untuk setiap bit ukuran, kita boleh membina satu ungkapan klasik tunggal yang mewakili XOR (pariti) semua hasil ukuran yang berkaitan. Ungkapan ini kemudian digunakan sebagai syarat dalam satu blok `if_test`, membolehkan Gate pembetulan digunakan pada kedalaman tetap. Pendekatan ini bukan sahaja memudahkan litar, malah memastikan pembetulan feedforward tidak menambah latensi tambahan yang tidak perlu.

In [6]:
def apply_ffwd_corrections(qc):
    control = 0  # control qubit
    target = qc.num_qubits - 1  # target qubit
    n = qc.num_qubits - 2  # number of qubits between target and control

    k = int(n / 2)
    x0 = check_even(n)

    if n > 1:
        _, c1, c2 = qc.cregs
    elif n > 0:
        _, c2 = qc.cregs

    # First, let's compute the parity of all ZZ measurements
    for i in range(k):
        if i == 0:
            parity_ZZ = expr.lift(
                c1[i]
            )  # Store the value of the first ZZ measurement in parity_ZZ
        else:
            parity_ZZ = expr.bit_xor(
                c1[i], parity_ZZ
            )  # Successively compute the parity via XOR operations

    for i in range(1, k + x0):
        if i == 1:
            parity_XX = expr.lift(
                c2[i - 1]
            )  # Store the value of the first XX measurement in parity_XX
        else:
            parity_XX = expr.bit_xor(
                c2[i - 1], parity_XX
            )  # Successively compute the parity via XOR operations

    if n > 0:
        with qc.if_test(parity_XX):
            qc.z(control)

    if n > 1:
        with qc.if_test(parity_ZZ):
            qc.x(target)
    return qc


qc = apply_ffwd_corrections(qc)
qc.draw(output="mpl", fold=-1, scale=0.5)

<Image src="../docs/images/tutorials/long-range-entanglement/extracted-outputs/4915791a-0.avif" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/tutorials/long-range-entanglement/extracted-outputs/4915791a-0.avif)

#### Ukur Qubit kawalan dan sasaran
Kita takrifkan fungsi pembantu yang membolehkan pengukuran Qubit kawalan dan sasaran dalam asas $XX$, $YY$, atau $ZZ$. Untuk mengesahkan keadaan Bell $(|00\rangle + |11\rangle)/\sqrt{2}$, nilai jangkaan $XX$ dan $ZZ$ sepatutnya kedua-duanya $+1$, kerana ia adalah penstabil bagi keadaan tersebut. Pengukuran $YY$ juga disokong di sini dan akan digunakan di bawah semasa mengira kesetiaan.

In [ ]:
def measure_in_basis(qc, basis="XX", add_barrier=True):
    control = 0  # control qubit
    target = qc.num_qubits - 1  # target qubit

    assert basis in ["XX", "YY", "ZZ"]

    qc = (
        qc.copy()
    )  # We copy the circuit because we want to measure in different bases
    cr = qc.cregs[0]

    if add_barrier:
        qc.barrier()

    if basis == "XX":
        qc.h(control)
        qc.h(target)
    elif basis == "YY":
        qc.sdg(control)
        qc.sdg(target)
        qc.h(control)
        qc.h(target)

    qc.measure(control, cr[0])
    qc.measure(target, cr[1])
    return qc


qc_YY = measure_in_basis(qc.copy(), basis="YY")
qc_YY.draw(
    output="mpl", fold=-1, scale=0.5
)  # Circuit for measuring in the YY basis

<Image src="../docs/images/tutorials/long-range-entanglement/extracted-outputs/d087d7c1-0.avif" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/tutorials/long-range-entanglement/extracted-outputs/d087d7c1-0.avif)

#### Gabungkan semuanya
Kita gabungkan pelbagai langkah yang ditakrifkan di atas untuk mencipta Gate CX jarak jauh di dua hujung garisan satu dimensi (1D). Langkah-langkahnya adalah seperti berikut:
- Memulakan Qubit kawalan dalam $|+\rangle$
- Menyediakan pasangan Bell
- Mengukur pasangan Qubit jiran
- Menggunakan pembetulan feedforward bergantung pada MCM

In [ ]:
def lrcx(distance, prep_barrier=True, pre_measure_barrier=True):
    qc = initialize_circuit(distance)
    qc = prepare_bell_pairs(qc, prep_barrier)
    qc = measure_bell_basis(qc, pre_measure_barrier)
    qc = apply_ffwd_corrections(qc)
    return qc


qc = lrcx(distance)
# Apply the measurement in the XX, YY, and ZZ bases
qc_XX, qc_YY, qc_ZZ = [
    measure_in_basis(qc, basis=basis) for basis in ["XX", "YY", "ZZ"]
]

qc_YY.draw(
    output="mpl", fold=-1, scale=0.5
)  # Circuit for measuring in the YY basis

<Image src="../docs/images/tutorials/long-range-entanglement/extracted-outputs/11fc8adc-0.avif" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/tutorials/long-range-entanglement/extracted-outputs/11fc8adc-0.avif)
#### Pelaksanaan berasaskan unitari dengan menukar Qubit ke tengah
Untuk perbandingan, kita periksa dulu kes di mana Gate CNOT jarak jauh dilaksanakan menggunakan sambungan jadi-jiran terdekat dan Gate unitari. Dalam rajah berikut, sebelah kiri adalah Circuit untuk Gate CNOT jarak jauh yang merentangi rantaian 1D sebanyak n-Qubit yang tertakluk kepada sambungan jadi-jiran terdekat sahaja. Di tengah pula adalah penguraian unitari yang setara yang boleh dilaksanakan dengan Gate CNOT setempat, dengan kedalaman Circuit $O(n)$.

![Long-range CNOT circuit](../docs/images/tutorials/long-range-entanglement/dynamic_vs_unitary_long_range_illustration.avif)

Litar di tengah boleh dilaksanakan seperti berikut:

In [22]:
def cnot_unitary(distance):
    """Generate a long range CNOT gate using local CNOTs on a 1D chain of qubits subject to n
    nearest-neighbor connections only.


    Args:
        distance (int) : The distance of the CNOT gate, with the convention that a distance of 0 is a nearest-neighbor CNOT.

    Returns:
        QuantumCircuit: A Quantum Circuit implementing a long-range CNOT gate between qubit 0 and qubit distance+1
    """
    assert distance >= 0
    n = distance  # number of qubits between target and control

    qr = QuantumRegister(
        n + 2, name="q"
    )  # Circuit with n qubits between control and target
    cr = ClassicalRegister(
        2, name="cr"
    )  # Classical register for measuring control and target qubits

    qc = QuantumCircuit(qr, cr, name="CNOT_unitary")

    control_qubit = 0

    qc.h(control_qubit)  # Prepare the control qubit in the |+> state

    k = int(n / 2)
    qc.barrier()
    for i in range(control_qubit, control_qubit + k):
        qc.cx(i, i + 1)
        qc.cx(i + 1, i)
        qc.cx(-i - 1, -i - 2)
        qc.cx(-i - 2, -i - 1)
    if n % 2 == 1:
        qc.cx(k + 2, k + 1)
        qc.cx(k + 1, k + 2)
    qc.barrier()
    qc.cx(k, k + 1)
    for i in range(control_qubit, control_qubit + k):
        qc.cx(k - i, k - 1 - i)
        qc.cx(k - 1 - i, k - i)
        qc.cx(k + i + 1, k + i + 2)
        qc.cx(k + i + 2, k + i + 1)
    if n % 2 == 1:
        qc.cx(-2, -1)
        qc.cx(-1, -2)

    return qc


qc_uni = cnot_unitary(distance)

Sekarang bina Circuit yang mengukur dalam asas $XX$, $YY$, dan $ZZ$, sama seperti yang kita buat untuk Circuit dinamik di atas.

In [ ]:
# Apply the measurement in the XX, YY, and ZZ bases
qc_uni_XX, qc_uni_YY, qc_uni_ZZ = [
    measure_in_basis(qc_uni, basis=basis) for basis in ["XX", "YY", "ZZ"]
]

qc_uni_YY.draw(
    output="mpl", fold=-1, scale=0.5
)  # Circuit for measuring in the YY basis

<Image src="../docs/images/tutorials/long-range-entanglement/extracted-outputs/b899e143-0.avif" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/tutorials/long-range-entanglement/extracted-outputs/b899e143-0.avif)

Sekarang kita telah membina kedua-dua litar dinamik dan litar uniter untuk contoh skala kecil dengan `distance=6`, kita kemudian transpilkan untuk dijalankan dahulu pada simulator tanpa hingar.

In [10]:
from qiskit_aer import AerSimulator

aer_backend = AerSimulator()
pm_sim = generate_preset_pass_manager(
    optimization_level=0, backend=aer_backend
)

# Dynamic circuits
isa_sim_dyn = pm_sim.run([qc_XX, qc_YY, qc_ZZ])

# Unitary circuits
isa_sim_uni = pm_sim.run([qc_uni_XX, qc_uni_YY, qc_uni_ZZ])

### Langkah 3: Laksanakan menggunakan primitif Qiskit
Kita kini boleh menjalankan eksperimen pada backend simulator tanpa hingar. Kita gunakan Qiskit Runtime Sampler dengan AerSimulator sebagai mod backend untuk melaksanakan litar.

In [13]:
sampler_sim = Sampler(mode=aer_backend)
sim_job = sampler_sim.run(isa_sim_dyn + isa_sim_uni)
sim_results = sim_job.result()

### Langkah 4: Proses selepas dan kembalikan keputusan dalam format klasikal yang dikehendaki

Setelah eksperimen berjaya dilaksanakan, kita kini memproses kiraan pengukuran untuk mengekstrak metrik yang bermakna.
Dalam langkah ini, kita lakukan perkara berikut:

- Takrifkan metrik kualiti untuk menilai prestasi CX jarak jauh.
- Kira nilai jangkaan operator Pauli daripada hasil pengukuran mentah.
- Gunakan nilai-nilai ini untuk mengira kesetiaan keadaan Bell yang dijana.

Dalam simulasi tanpa hingar, kita akan sahkan bahawa metrik kesetiaan adalah $1$ untuk litar yang dibina. Dalam eksperimen pada QPU sebenar, analisis ini akan memberi gambaran jelas tentang seberapa baik Circuit dinamik berprestasi berbanding pelaksanaan garis dasar uniter.

#### Metrik kualiti

Untuk menilai kejayaan protokol CX jarak jauh, kita ukur sejauh mana keadaan output menghampiri keadaan Bell yang ideal. Cara yang mudah untuk mengukur ini ialah dengan mengira kesetiaan keadaan menggunakan nilai jangkaan operator Pauli. Kita boleh mengira kesetiaan untuk keadaan Bell pada keadaan kawalan dan sasaran selepas mengetahui $\braket{XX}$, $\braket{YY}$, dan $\braket{ZZ}$. Khususnya,

$$ F = \frac{1}{4} (1 + \braket{XX} - \braket{YY} + \braket{ZZ})$$

Untuk mengira nilai jangkaan ini daripada data pengukuran mentah, kita takrifkan satu set fungsi pembantu:

- **`compute_ZZ_expectation`**: Diberi kiraan pengukuran, mengira nilai jangkaan operator Pauli dua-Qubit dalam asas $Z$.
- **`compute_fidelity`**: Menggabungkan nilai jangkaan $XX$, $YY$, dan $ZZ$ ke dalam ungkapan kesetiaan di atas.
- **`get_counts_from_bitarray`**: Utiliti untuk mengekstrak kiraan daripada objek keputusan Backend.

In [36]:
def compute_ZZ_expectation(counts):
    total = sum(counts.values())
    expectation = 0
    for bitstring, count in counts.items():
        # Ensure bitstring is 2 bits
        z1 = (-1) ** (int(bitstring[-1]))
        z2 = (-1) ** (int(bitstring[-2]))
        expectation += z1 * z2 * count
    return expectation / total


def compute_fidelity(counts_xx, counts_yy, counts_zz):
    xx, yy, zz = [
        compute_ZZ_expectation(c) for c in [counts_xx, counts_yy, counts_zz]
    ]
    return 1 / 4 * (1 + xx - yy + zz)

In [37]:
# Dynamic fidelity
counts_xx = sim_results[0].data.cr.get_counts()
counts_yy = sim_results[1].data.cr.get_counts()
counts_zz = sim_results[2].data.cr.get_counts()
fidelity_dyn = compute_fidelity(counts_xx, counts_yy, counts_zz)

# Unitary fidelity
counts_xx = sim_results[3].data.cr.get_counts()
counts_yy = sim_results[4].data.cr.get_counts()
counts_zz = sim_results[5].data.cr.get_counts()
fidelity_uni = compute_fidelity(counts_xx, counts_yy, counts_zz)

print(f"Dynamic fidelity (distance={distance}): {fidelity_dyn:.4f}")
print(f"Unitary fidelity (distance={distance}): {fidelity_uni:.4f}")

Dynamic fidelity (distance=6): 1.0000
Unitary fidelity (distance=6): 1.0000


As expected in a noiseless simulation, the fidelities in both dynamic circuits and unitary circuits are $1$.

## Large-scale hardware example

Here we now put all of these details together into a single workflow at a larger scale, which is then run on real quantum hardware.

### Generate circuits for different distances

We now generate long-range CX circuits for a range of qubit separations up to 60 qubits apart. For each distance, we build circuits that measure in the $XX$, $YY$, and $ZZ$ bases, which will later be used to compute fidelities.

The list of distances includes both short- and long-range separations, with `distance = 0` corresponding to a nearest-neighbor CX. These same distances will also be used to generate the corresponding unitary circuits later for comparison.

In [ ]:
# -------------------------Step 1-------------------------
distances = [
    0,
    1,
    2,
    3,
    6,
    11,
    16,
    21,
    28,
    35,
    44,
    55,
    60,
]  # Distances for long range CX. distance of 0 is a nearest-neighbor CX
distances.sort()
assert min(distances) >= 0
basis_list = ["XX", "YY", "ZZ"]

# Dynamic circuits
circuits_dyn = []
for distance in distances:
    for basis in basis_list:
        circuits_dyn.append(
            measure_in_basis(lrcx(distance, prep_barrier=False), basis=basis)
        )
print(f"Number of circuits: {len(circuits_dyn)}")

# Unitary circuits
circuits_uni = []
for distance in distances:
    for basis in basis_list:
        circuits_uni.append(
            measure_in_basis(cnot_unitary(distance), basis=basis)
        )

print(f"Number of circuits: {len(circuits_uni)}")

Seperti yang dijangkakan dalam simulasi tanpa hingar, kesetiaan dalam kedua-dua litar dinamik dan litar uniter adalah $1$.
## Contoh perkakasan skala besar
Di sini kita gabungkan semua butiran ini ke dalam satu aliran kerja berskala lebih besar, yang kemudian dijalankan pada perkakasan kuantum sebenar.
### Jana Circuit untuk Jarak Berbeza
Kita sekarang akan jana Circuit CX jarak jauh untuk pelbagai jarak pemisahan Qubit sehingga 60 Qubit jaraknya. Untuk setiap jarak, kita bina Circuit yang mengukur dalam asas $XX$, $YY$, dan $ZZ$, yang akan digunakan kemudian untuk mengira fideliti.

Senarai jarak ini merangkumi pemisahan jarak dekat dan jarak jauh, dengan `distance = 0` bersamaan dengan CX jadi-jiran terdekat. Jarak yang sama ini juga akan digunakan untuk menjana Circuit unitari yang sepadan kemudian bagi tujuan perbandingan.

In [24]:
# -------------------------Step 2-------------------------
# Set up access to IBM Quantum devices
from qiskit.circuit import IfElseOp

service = QiskitRuntimeService()
backend = service.least_busy(
    operational=True, simulator=False, min_num_qubits=156
)

Sekarang kita dah ada Circuit dinamik dan unitari untuk pelbagai jarak, kita dah bersedia untuk transpilasi. Kita perlu pilih peranti Backend dahulu.

In [25]:
if "if_else" not in backend.target.operation_names:
    backend.target.add_instruction(IfElseOp, name="if_else")

#### Use Layer Fidelity string for selecting 1D chain
Since we want to compare the performance of dynamic and unitary circuits on a 1D chain, we use the Layer Fidelity string to select a linear topology of the best chain of qubits from the device. This ensures that both types of circuits are transpiled under the same connectivity constraints, allowing for a fair comparison of their performance.

In [26]:
# This selects best qubits for longest distance and uses the same control for all lengths
lf_qubits = backend.properties().to_dict()[
    "general_qlists"
]  # best linear chain qubits
chosen_layouts = {
    distance: [
        val["qubits"]
        for val in lf_qubits
        if val["name"] == f"lf_{distances[-1] + 2}"
    ][0][: distance + 2]
    for distance in distances
}
print(chosen_layouts[max(distances)])  # best qubits at each distance

[11, 12, 13, 14, 15, 19, 35, 34, 33, 39, 53, 54, 55, 59, 75, 74, 73, 72, 71, 70, 69, 68, 67, 66, 65, 64, 63, 62, 61, 76, 81, 82, 83, 84, 85, 86, 87, 97, 107, 108, 109, 110, 111, 98, 91, 92, 93, 94, 95, 99, 115, 114, 113, 119, 133, 132, 131, 138, 151, 150, 149, 148]


In [27]:
isa_circuits_dyn = []
isa_circuits_uni = []

# Using the same initial layouts for both circuits for better apples to apples comparison
for qc in circuits_dyn:
    pm = generate_preset_pass_manager(
        optimization_level=1,
        backend=backend,
        initial_layout=chosen_layouts[qc.num_qubits - 2],
    )
    isa_circuits_dyn.append(pm.run(qc))

for qc in circuits_uni:
    pm = generate_preset_pass_manager(
        optimization_level=1,
        backend=backend,
        initial_layout=chosen_layouts[qc.num_qubits - 2],
    )
    isa_circuits_uni.append(pm.run(qc))

In [28]:
print(
    f"2Q depth: {isa_circuits_dyn[14].depth(lambda x: x.operation.num_qubits == 2)}"
)
isa_circuits_dyn[14].draw("mpl", fold=-1, idle_wires=0)

2Q depth: 2


<Image src="../docs/images/tutorials/long-range-entanglement/extracted-outputs/c77c3fd3-1.avif" alt="Output of the previous code cell" />

In [29]:
print(
    f"2Q depth: {isa_circuits_uni[14].depth(lambda x: x.operation.num_qubits == 2)}"
)
isa_circuits_uni[14].draw("mpl", fold=-1, idle_wires=False)

2Q depth: 13


<Image src="../docs/images/tutorials/long-range-entanglement/extracted-outputs/7e5fc240-1.avif" alt="Output of the previous code cell" />

### Visualize qubits used for the LRCX circuit

In this section, we examine how the LRCX circuit is mapped onto hardware. We start by visualizing the physical qubits used in the circuit and then study how the control–target distance in the layout impacts the number of operations.

In [30]:
# Note: the qubit coordinates must be hard-coded.
# The backend API does not currently provide this information directly.
# If using a different backend, you will need to adjust the coordinates accordingly,
# or set the qubit_coordinates = None to use the default layout coordinates.


def _heron_coords_r2():
    """Generate coordinates for the Heron layout in R2. Note"""
    cord_map = np.array(
        [
            [
                0,
                1,
                2,
                3,
                4,
                5,
                6,
                7,
                8,
                9,
                10,
                11,
                12,
                13,
                14,
                15,
                3,
                7,
                11,
                15,
                0,
                1,
                2,
                3,
                4,
                5,
                6,
                7,
                8,
                9,
                10,
                11,
                12,
                13,
                14,
                15,
                1,
                5,
                9,
                13,
                0,
                1,
                2,
                3,
                4,
                5,
                6,
                7,
                8,
                9,
                10,
                11,
                12,
                13,
                14,
                15,
                3,
                7,
                11,
                15,
                0,
                1,
                2,
                3,
                4,
                5,
                6,
                7,
                8,
                9,
                10,
                11,
                12,
                13,
                14,
                15,
                1,
                5,
                9,
                13,
                0,
                1,
                2,
                3,
                4,
                5,
                6,
                7,
                8,
                9,
                10,
                11,
                12,
                13,
                14,
                15,
                3,
                7,
                11,
                15,
                0,
                1,
                2,
                3,
                4,
                5,
                6,
                7,
                8,
                9,
                10,
                11,
                12,
                13,
                14,
                15,
                1,
                5,
                9,
                13,
                0,
                1,
                2,
                3,
                4,
                5,
                6,
                7,
                8,
                9,
                10,
                11,
                12,
                13,
                14,
                15,
                3,
                7,
                11,
                15,
                0,
                1,
                2,
                3,
                4,
                5,
                6,
                7,
                8,
                9,
                10,
                11,
                12,
                13,
                14,
                15,
            ],
            -1
            * np.array([j for i in range(15) for j in [i] * [16, 4][i % 2]]),
        ],
        dtype=int,
    )

    hcords = []
    ycords = cord_map[0]
    xcords = cord_map[1]
    for i in range(156):
        hcords.append([xcords[i] + 1, np.abs(ycords[i]) + 1])

    return hcords


# Visualize the active qubits in the circuit layout
plot_circuit_layout(
    circuit=isa_circuits_uni[-1],
    backend=backend,
    view="physical",
    qubit_coordinates=_heron_coords_r2(),
)

<Image src="../docs/images/tutorials/long-range-entanglement/extracted-outputs/2d090f8a-0.avif" alt="Output of the previous code cell" />

Next, we execute the experiment on the real backend. We also make use of batching to efficiently run the experiment across multiple trials. Running repeated trials allows us to compute averages for a more accurate comparison between the unitary and dynamic methods, as well as to quantify their variability by comparing the deviations across runs.

In [31]:
# -------------------------Step 3-------------------------
num_trials = 10
jobs_uni = []
jobs_dyn = []
with Batch(backend=backend) as batch:
    sampler = Sampler(mode=batch)
    sampler.options.environment.job_tags = ["TUT_LRE"]
    for _ in range(num_trials):
        jobs_uni.append(sampler.run(isa_circuits_uni, shots=1024))
        jobs_dyn.append(sampler.run(isa_circuits_dyn, shots=1024))

We compute the fidelity for the dynamic long-range CX circuits. For each distance, we extract measurement outcomes in the $\braket{XX}$, $\braket{YY}$, and $\braket{ZZ}$ bases. These results are combined using the previously defined helper functions to calculate the fidelity according to  $F = \tfrac{1}{4} \big( 1 + \langle XX \rangle - \langle YY \rangle + \langle ZZ \rangle \big)$. This provides the observed fidelity of the dynamically executed protocol at each distance.

In [32]:
# -------------------------Step 4-------------------------
fidelities_dyn = []

# loop over trials
for job in jobs_dyn:
    result_dyn = job.result()
    trial_fidelities = []
    # loop over all distances
    for ind, dist in enumerate(distances):
        counts_xx = result_dyn[ind * 3].data.cr.get_counts()
        counts_yy = result_dyn[ind * 3 + 1].data.cr.get_counts()
        counts_zz = result_dyn[ind * 3 + 2].data.cr.get_counts()
        trial_fidelities.append(
            compute_fidelity(counts_xx, counts_yy, counts_zz)
        )
    fidelities_dyn.append(trial_fidelities)
# average over trials for each distance
avg_fidelities_dyn = np.mean(fidelities_dyn, axis=0)
std_fidelities_dyn = np.std(fidelities_dyn, axis=0)

Now we compute the fidelity for the unitary long-range CX circuits, and we do it the same way as we did for the dynamic circuits above.

In [33]:
fidelities_uni = []

# loop over trials
for job in jobs_uni:
    result_uni = job.result()
    trial_fidelities = []
    # loop over all distances
    for ind, dist in enumerate(distances):
        counts_xx = result_uni[ind * 3].data.cr.get_counts()
        counts_yy = result_uni[ind * 3 + 1].data.cr.get_counts()
        counts_zz = result_uni[ind * 3 + 2].data.cr.get_counts()
        trial_fidelities.append(
            compute_fidelity(counts_xx, counts_yy, counts_zz)
        )
    fidelities_uni.append(trial_fidelities)
# average over trials for each distance
avg_fidelities_uni = np.mean(fidelities_uni, axis=0)
std_fidelities_uni = np.std(fidelities_uni, axis=0)

![Output of the previous code cell](../docs/images/tutorials/long-range-entanglement/extracted-outputs/7e5fc240-1.avif)

### Visualisasikan Qubit yang digunakan untuk Circuit LRCX
Dalam bahagian ini, kita periksa bagaimana Circuit LRCX dipetakan ke perkakasan. Kita mulakan dengan memvisualisasikan Qubit fizikal yang digunakan dalam Circuit, kemudian kaji bagaimana jarak kawalan–sasaran dalam susun atur mempengaruhi bilangan operasi.

Seterusnya, kita jalankan eksperimen pada backend sebenar. Kita juga guna batching untuk menjalankan eksperimen secara cekap merentas beberapa percubaan. Menjalankan percubaan berulang membolehkan kita mengira purata untuk perbandingan yang lebih tepat antara kaedah uniter dan dinamik, serta untuk mengukur kebolehubahan dengan membandingkan sisihan merentas setiap cubaan.

In [34]:
fig, ax = plt.subplots()

# Unitary with error bars
ax.errorbar(
    distances,
    avg_fidelities_uni,
    yerr=std_fidelities_uni,
    fmt="o-.",
    color="c",
    ecolor="c",
    elinewidth=1,
    capsize=4,
    label="Unitary",
)
# Dynamic with error bars
ax.errorbar(
    distances,
    avg_fidelities_dyn,
    yerr=std_fidelities_dyn,
    fmt="o-.",
    color="m",
    ecolor="m",
    elinewidth=1,
    capsize=4,
    label="Dynamic",
)
# Random gate baseline
ax.axhline(y=1 / 4, linestyle="--", color="gray", label="Random gate")

legend = ax.legend(frameon=True)
for text in legend.get_texts():
    text.set_color("black")
legend.get_frame().set_facecolor("white")
legend.get_frame().set_edgecolor("black")
ax.set_title(
    "Bell State Fidelity vs Control–Target Separation", color="black"
)
ax.set_xlabel("Distance", color="black")
ax.set_ylabel("Bell state fidelity", color="black")
ax.grid(linestyle=":", linewidth=0.6, alpha=0.4, color="gray")
ax.set_ylim((0.2, 1))
ax.set_facecolor("white")
fig.patch.set_facecolor("white")
for spine in ax.spines.values():
    spine.set_visible(True)
    spine.set_color("black")
ax.tick_params(axis="x", colors="black")
ax.tick_params(axis="y", colors="black")
plt.show()

<Image src="../docs/images/tutorials/long-range-entanglement/extracted-outputs/724da22d-0.avif" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/tutorials/long-range-entanglement/extracted-outputs/2d090f8a-0.avif)

Seterusnya, kita jalankan eksperimen pada backend sebenar. Kita juga guna batching untuk menjalankan eksperimen secara cekap merentas beberapa percubaan. Menjalankan percubaan berulang membolehkan kita mengira purata untuk perbandingan yang lebih tepat antara kaedah uniter dan dinamik, serta untuk mengukur kebolehubahan dengan membandingkan sisihan merentas setiap cubaan.

In [35]:
# Compute metrics for each distance, skipping the basis circuits since they are identical for each distance
depths_2q_dyn = [
    c.depth(lambda x: x.operation.num_qubits == 2)
    for c in isa_circuits_dyn[::3]
]
meas_dyn = [
    sum(1 for instr in c.data if instr.operation.name == "measure")
    for c in isa_circuits_dyn[::3]
]

depths_2q_uni = [
    c.depth(lambda x: x.operation.num_qubits == 2)
    for c in isa_circuits_uni[::3]
]
meas_uni = [
    sum(1 for instr in c.data if instr.operation.name == "measure")
    for c in isa_circuits_uni[::3]
]

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

axes[0].plot(
    distances, depths_2q_uni, "o-.", color="c", label="Unitary (2Q depth)"
)
axes[0].plot(
    distances, depths_2q_dyn, "o-.", color="m", label="Dynamic (2Q depth)"
)
axes[0].set_xlabel("Number of qubits between control and target")
axes[0].set_ylabel("Two-qubit depth")
axes[0].grid(True, linestyle=":", linewidth=0.6, alpha=0.4)
axes[0].legend()

axes[1].plot(
    distances, meas_uni, "o-.", color="c", label="Unitary (# measurements)"
)
axes[1].plot(
    distances, meas_dyn, "o-.", color="m", label="Dynamic (# measurements)"
)
axes[1].set_xlabel("Number of qubits between control and target")
axes[1].set_ylabel("Number of measurements")
axes[1].grid(True, linestyle=":", linewidth=0.6, alpha=0.4)
axes[1].legend()

fig.suptitle("Scaling of Unitary vs Dynamic LRCX with Distance", fontsize=12)

plt.tight_layout()
plt.show()

<Image src="../docs/images/tutorials/long-range-entanglement/extracted-outputs/3dcff343-0.avif" alt="Output of the previous code cell" />

Kita kira kesetiaan untuk Circuit CX jarak jauh dinamik. Bagi setiap jarak, kita ekstrak hasil pengukuran dalam asas $\braket{XX}$, $\braket{YY}$, dan $\braket{ZZ}$. Keputusan-keputusan ini digabungkan menggunakan fungsi pembantu yang telah ditakrifkan sebelum ini untuk mengira kesetiaan mengikut $F = \tfrac{1}{4} \big( 1 + \langle XX \rangle - \langle YY \rangle + \langle ZZ \rangle \big)$. Ini memberikan kesetiaan yang diperhatikan bagi protokol yang dilaksanakan secara dinamik pada setiap jarak.